In [1]:
from databricks.connect import DatabricksSession
spark = DatabricksSession.builder.serverless().getOrCreate()

In [2]:
from pyspark.sql import functions as F, Window

# ── Source paths ────────────────────────────────────────────────────────────
CUSTOMERS_PATH           = "/Volumes/northstar_dev/raw/landing/customers"
MERCHANTS_PATH           = "/Volumes/northstar_dev/raw/landing/merchants"
TRANSACTIONS_PATH        = "/Volumes/northstar_dev/raw/landing/transactions"
DEVICE_SESSION_LOGS_PATH = "/Volumes/northstar_dev/raw/landing/device_session_logs"

# ── Cast helpers (raw landing zone — all columns land as string) ───────────────
def _date(c):   return F.to_date(F.col(c), "yyyy-MM-dd")
def _ts(c):     return F.to_timestamp(F.col(c))
def _double(c): return F.col(c).cast("double")
def _int(c):    return F.col(c).cast("integer")

# ── Readers ────────────────────────────────────────────────────────────────
def read_csv(path):
    return (spark.read.format("csv")
            .option("header", True)
            .option("recursiveFileLookup", "true")
            .load(path))

def read_json(path):
    return (spark.read.format("json")
            .option("recursiveFileLookup", "true")
            .load(path))

print("Constants and helpers loaded.")

Constants and helpers loaded.


# NorthstarPay — Blocker & Warning Investigation

Root-cause investigation of all findings flagged in the EDA scorecard. Run cells top-to-bottom — each section is self-contained after the shared setup cell.

| Finding | Severity | Count | Context |
|---|---|---|---|
| Pre-signup transactions | BLOCKER | 107,232 | ~20% of transaction volume |
| Impossible geographic velocity | BLOCKER | 1,019 | session pairs across device_session_logs |
| Amount outliers (top 0.1%) | WARNING | 573 | transactions above 99.9th percentile |

In [3]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DoubleType, DateType
from datetime import date

# ── Volume paths ───────────────────────────────────────────────────────────
CUSTOMERS_PATH           = "/Volumes/northstar_dev/raw/landing/customers"
MERCHANTS_PATH           = "/Volumes/northstar_dev/raw/landing/merchants"
TRANSACTIONS_PATH        = "/Volumes/northstar_dev/raw/landing/transactions"
DEVICE_SESSION_LOGS_PATH = "/Volumes/northstar_dev/raw/landing/device_session_logs"

# ── Cast helpers ───────────────────────────────────────────────────────────
def _date(c):   return F.to_date(F.col(c), "yyyy-MM-dd")
def _ts(c):     return F.to_timestamp(F.col(c))
def _double(c): return F.col(c).cast(DoubleType())

TODAY = F.lit(str(date.today()))

# ── Load DataFrames ───────────────────────────────────────────────────────────
def _csv(path):
    return (spark.read.format("csv")
            .option("header", True)
            .option("recursiveFileLookup", "true")
            .load(path))

df_customers    = _csv(CUSTOMERS_PATH)
df_merchants    = _csv(MERCHANTS_PATH)
df_transactions = _csv(TRANSACTIONS_PATH)
df_devices      = (spark.read.format("json")
                   .option("recursiveFileLookup", "true")
                   .load(DEVICE_SESSION_LOGS_PATH))

all_cols = {
    "customers":    df_customers.columns,
    "merchants":    df_merchants.columns,
    "transactions": df_transactions.columns,
    "devices":      df_devices.columns,
}

print("DataFrames loaded:")
print(f"  customers           : {df_customers.count():,}")
print(f"  merchants           : {df_merchants.count():,}")
print(f"  transactions        : {df_transactions.count():,}")
print(f"  device_session_logs : {df_devices.count():,}")

DataFrames loaded:
  customers           : 50,000
  merchants           : 9,000
  transactions        : 525,292
  device_session_logs : 261,414


## Blocker 1: Pre-signup Transactions (107,232 rows, ~20% of volume)

Transactions where `txn_date < signup_date` for the same customer. At 20% of volume this is systemic, not a handful of edge cases. Goal here is to determine whether this is a **data capture bug, a migration artifact, or a legitimate business scenario** before deciding how to handle it.

Investigation sequence:
1. **Sample inspection** — pull 30 worst-gap customers, inspect `signup_date` vs earliest transaction manually
2. **Concentration check** — spread evenly across segments/risk bands, or clustered in one?
3. **Time gap distribution** — 1-day gaps suggest timezone/rounding; multi-year gaps confirm a real data problem
4. **`last_updated_ts` check** — if `signup_date` was overwritten by a system migration, `last_updated_ts` will be suspiciously recent relative to the earliest transaction date

In [0]:
# ── Blocker 1a: Sample Pre-Signup Customers ─────────────────────────────────
# Build the full pre-signup DataFrame — reused by Blocker 1b, 1c, 1d below.

pre_signup_df = (
    df_transactions
    .select(
        "transaction_id", "customer_id",
        _date("txn_date").alias("txn_d"),
        _ts("txn_timestamp").alias("txn_ts"),
    )
    .join(
        df_customers.select(
            "customer_id", "segment", "risk_band", "kyc_status",
            _date("signup_date").alias("signup_d"),
            _ts("last_updated_ts").alias("last_updated"),
        ),
        on="customer_id",
        how="inner"
    )
    .filter(F.col("txn_d") < F.col("signup_d"))
    .withColumn("days_before_signup", F.datediff(F.col("signup_d"), F.col("txn_d")))
)

total_pre_signup = pre_signup_df.count()
affected_customers = pre_signup_df.select("customer_id").distinct().count()
print(f"Pre-signup transactions    : {total_pre_signup:,}")
print(f"Distinct affected customers: {affected_customers:,}")

print("\n--- Sample: 30 customers with largest signup gap ---")
display(
    pre_signup_df
    .select("customer_id", "signup_d", "txn_d", "days_before_signup",
            "segment", "risk_band", "kyc_status", "last_updated")
    .orderBy(F.desc("days_before_signup"))
    .limit(30)
)

Pre-signup transactions    : 107,232
Distinct affected customers: 14,598

--- Sample: 30 customers with largest signup gap ---


customer_id,signup_d,txn_d,days_before_signup,segment,risk_band,kyc_status,last_updated
CUST-0022308,2026-06-28,2025-04-07,447,retail,low,verified,2026-06-29T00:00:00.000Z
CUST-0045791,2026-06-26,2025-04-06,446,retail,medium,verified,2026-06-28T00:00:00.000Z
CUST-0024185,2026-06-28,2025-04-08,446,retail,low,verified,2026-06-29T00:00:00.000Z
CUST-0046714,2026-06-27,2025-04-08,445,business,low,pending,2026-06-28T00:00:00.000Z
CUST-0041036,2026-06-25,2025-04-06,445,premium,low,verified,2026-06-25T00:00:00.000Z
CUST-0038725,2026-06-27,2025-04-08,445,business,low,verified,2026-06-29T00:00:00.000Z
CUST-0032800,2026-06-28,2025-04-09,445,premium,medium,verified,2026-06-28T00:00:00.000Z
CUST-0035583,2026-06-26,2025-04-08,444,retail,low,verified,2026-06-26T00:00:00.000Z
CUST-0013638,2026-06-28,2025-04-10,444,retail,low,verified,2026-06-28T00:00:00.000Z
CUST-0045752,2026-06-27,2025-04-09,444,premium,low,verified,2026-06-27T00:00:00.000Z


In [0]:
# ── Blocker 1b: Concentration by Segment and Risk Band ─────────────────────
# If concentrated in one segment, it may be a legitimate business reason
# (e.g. migrated customers). If evenly spread, it is almost certainly systemic.

print("--- By segment ---")
display(
    pre_signup_df.groupBy("segment")
    .agg(
        F.countDistinct("customer_id").alias("affected_customers"),
        F.count("transaction_id").alias("affected_txns"),
    )
    .join(
        df_customers.groupBy("segment").count().withColumnRenamed("count", "total_customers"),
        on="segment", how="left"
    )
    .withColumn("affected_%", F.round(F.col("affected_customers") / F.col("total_customers") * 100, 2))
    .orderBy(F.desc("affected_txns"))
)

print("\n--- By risk_band ---")
display(
    pre_signup_df.groupBy("risk_band")
    .agg(
        F.countDistinct("customer_id").alias("affected_customers"),
        F.count("transaction_id").alias("affected_txns"),
    )
    .join(
        df_customers.groupBy("risk_band").count().withColumnRenamed("count", "total_customers"),
        on="risk_band", how="left"
    )
    .withColumn("affected_%", F.round(F.col("affected_customers") / F.col("total_customers") * 100, 2))
    .orderBy(F.desc("affected_txns"))
)

print("\n--- By kyc_status ---")
display(
    pre_signup_df.groupBy("kyc_status")
    .agg(
        F.countDistinct("customer_id").alias("affected_customers"),
        F.count("transaction_id").alias("affected_txns"),
    )
    .orderBy(F.desc("affected_txns"))
)

--- By segment ---


segment,affected_customers,affected_txns,total_customers,affected_%
retail,10252,76183,34842,29.42
premium,3266,23402,11200,29.16
business,1080,7647,3958,27.29



--- By risk_band ---


risk_band,affected_customers,affected_txns,total_customers,affected_%
low,9521,70610,32611,29.2
medium,3623,26563,12404,29.21
high,1454,10059,4985,29.17



--- By kyc_status ---


kyc_status,affected_customers,affected_txns
verified,13157,96919
pending,1016,7285
rejected,425,3028


In [0]:
# ── Blocker 1c: Time Gap Distribution ────────────────────────────────────────
# Gap <= 1 day  → likely timezone/rounding  → fixable with a 1-day tolerance rule
# Gap > 1 year  → real data problem          → needs source-system investigation

print("--- Gap statistics ---")
gap_stats = pre_signup_df.select(
    F.min("days_before_signup").alias("min"),
    F.max("days_before_signup").alias("max"),
    F.round(F.mean("days_before_signup"), 1).alias("mean"),
    F.percentile_approx("days_before_signup", 0.5).alias("median"),
    F.percentile_approx("days_before_signup", 0.95).alias("p95"),
    F.percentile_approx("days_before_signup", 0.99).alias("p99"),
    F.count(F.when(F.col("days_before_signup") <= 1,   1)).alias("within_1_day"),
    F.count(F.when(F.col("days_before_signup") <= 7,   1)).alias("within_7_days"),
    F.count(F.when(F.col("days_before_signup") > 365,  1)).alias("over_1_year"),
).collect()[0]

for k, v in gap_stats.asDict().items():
    print(f"  {k:<20} : {v}")

print("\n--- Gap buckets ---")
display(
    pre_signup_df
    .withColumn("gap_bucket",
        F.when(F.col("days_before_signup") <= 1,   "0–1 days    (timezone/rounding)")
         .when(F.col("days_before_signup") <= 7,   "2–7 days    (possible lag)")
         .when(F.col("days_before_signup") <= 30,  "8–30 days   (investigate)")
         .when(F.col("days_before_signup") <= 365, "31–365 days (serious)")
         .otherwise("> 1 year           (blocker)")
    )
    .groupBy("gap_bucket")
    .count()
    .orderBy("gap_bucket")
)

--- Gap statistics ---
  min                  : 1
  max                  : 447
  mean                 : 149.0
  median               : 131
  p95                  : 347
  p99                  : 403
  within_1_day         : 468
  within_7_days        : 3416
  over_1_year          : 3634

--- Gap buckets ---


gap_bucket,count
0–1 days (timezone/rounding),468
2–7 days (possible lag),2948
31–365 days (serious),89558
8–30 days (investigate),10624
> 1 year (blocker),3634


In [0]:
# ── Blocker 1d: last_updated_ts Overwrite Check ────────────────────────────
# Hypothesis: signup_date was overwritten by a later system update (e.g. migration),
# making it look like customers signed up AFTER their first transaction.
# Indicator: last_updated_ts >> signup_date for affected customers.

earliest_txn = (
    df_transactions
    .select("customer_id", _date("txn_date").alias("txn_d"))
    .groupBy("customer_id")
    .agg(F.min("txn_d").alias("earliest_txn_date"))
)

overwrite_check = (
    df_customers
    .select(
        "customer_id",
        _date("signup_date").alias("signup_d"),
        _ts("last_updated_ts").alias("last_updated"),
    )
    .join(earliest_txn, on="customer_id", how="inner")
    .filter(F.col("earliest_txn_date") < F.col("signup_d"))  # affected customers only
    .withColumn("last_updated_date", F.col("last_updated").cast(DateType()))
    .withColumn("updated_after_signup",
        F.col("last_updated_date") > F.col("signup_d"))
    .withColumn("days_updated_after_signup",
        F.datediff(F.col("last_updated_date"), F.col("signup_d")))
)

stats = overwrite_check.select(
    F.count("customer_id").alias("total"),
    F.count(F.when(F.col("updated_after_signup"), 1)).alias("updated_after"),
    F.round(F.mean(
        F.when(F.col("updated_after_signup"), F.col("days_updated_after_signup"))
    ), 1).alias("avg_days_after_signup"),
).collect()[0]

pct_updated = round(stats["updated_after"] / stats["total"] * 100, 1) if stats["total"] > 0 else 0
print(f"  Affected customers                       : {stats['total']:,}")
print(f"  last_updated_ts AFTER signup_date        : {stats['updated_after']:,}  ({pct_updated}%)")
print(f"  Avg days between signup and last update  : {stats['avg_days_after_signup']}")
print()
print("  HIGH % → signup_date overwritten by a system update (migration artifact).")
print("  LOW %  → signup_date reflects something genuinely wrong at source.")

print("\n--- Sample: 20 customers ordered by days updated after signup ---")
display(
    overwrite_check
    .filter(F.col("updated_after_signup"))
    .orderBy(F.desc("days_updated_after_signup"))
    .limit(20)
)

  Affected customers                       : 14,598
  last_updated_ts AFTER signup_date        : 14,357  (98.3%)
  Avg days between signup and last update  : 94.5

  HIGH % → signup_date overwritten by a system update (migration artifact).
  LOW %  → signup_date reflects something genuinely wrong at source.

--- Sample: 20 customers ordered by days updated after signup ---


customer_id,signup_d,last_updated,earliest_txn_date,last_updated_date,updated_after_signup,days_updated_after_signup
CUST-0000769,2025-04-13,2026-06-27T00:00:00.000Z,2025-04-07,2026-06-27,true,440
CUST-0020911,2025-04-10,2026-06-23T00:00:00.000Z,2025-04-08,2026-06-23,true,439
CUST-0018883,2025-04-11,2026-06-18T00:00:00.000Z,2025-04-06,2026-06-18,true,433
CUST-0001858,2025-04-10,2026-06-16T00:00:00.000Z,2025-04-08,2026-06-16,true,432
CUST-0003375,2025-04-16,2026-06-09T00:00:00.000Z,2025-04-13,2026-06-09,true,419
CUST-0043582,2025-05-04,2026-06-24T00:00:00.000Z,2025-04-10,2026-06-24,true,416
CUST-0013632,2025-04-24,2026-06-12T00:00:00.000Z,2025-04-12,2026-06-12,true,414
CUST-0047042,2025-04-21,2026-06-09T00:00:00.000Z,2025-04-16,2026-06-09,true,414
CUST-0047440,2025-04-28,2026-06-14T00:00:00.000Z,2025-04-07,2026-06-14,true,412
CUST-0037610,2025-05-03,2026-06-19T00:00:00.000Z,2025-04-21,2026-06-19,true,412


## Blocker 2: Impossible Geographic Velocity (1,019 session pairs)

Session pairs where implied travel speed between consecutive events for the same customer exceeds 900 km/h. Could be:
- Data noise: low GPS accuracy or VPN/proxy masking real location
- App/OS bug: specific app version reporting wrong coordinates
- Real signal: account takeover, device sharing, or confirmed fraud

Investigation sequence:
1. **Accuracy check** — pairs with high `accuracy_m` are imprecise locations, not real impossible jumps
2. **Concentration check** — clustered on a few `device_id`s or `app_version`s (bug) vs spread thin (signal)
3. **Fraud correlation** — do velocity-flagged customers have materially higher fraud rates than the rest?

In [0]:
# ── Blocker 2: Rebuild Velocity Pairs with Full Context ──────────────────────
# Includes accuracy_m, app_version, os on both sides of each pair for diagnosis.

MAX_SPEED_KMH   = 900
EARTH_RADIUS_KM = 6371

df_geo = (
    df_devices
    .filter(
        F.col("geo").isNotNull() &
        F.col("geo.lat").isNotNull() &
        F.col("geo.long").isNotNull()
    )
    .select(
        "customer_id", "session_id", "device_id",
        F.col("device_meta.app_version").alias("app_version"),
        F.col("device_meta.os").alias("os"),
        _ts("event_timestamp").alias("event_ts"),
        F.col("geo.lat").alias("lat"),
        F.col("geo.long").alias("lng"),
        F.col("geo.accuracy_m").alias("accuracy_m"),
    )
)

w = Window.partitionBy("customer_id").orderBy("event_ts")

df_velocity = (
    df_geo
    .withColumn("prev_lat",         F.lag("lat",         1).over(w))
    .withColumn("prev_lng",         F.lag("lng",         1).over(w))
    .withColumn("prev_ts",          F.lag("event_ts",    1).over(w))
    .withColumn("prev_accuracy_m",  F.lag("accuracy_m",  1).over(w))
    .withColumn("prev_app_version", F.lag("app_version", 1).over(w))
    .withColumn("prev_os",          F.lag("os",          1).over(w))
    .filter(F.col("prev_lat").isNotNull())
    .withColumn("dist_km",
        2 * EARTH_RADIUS_KM * F.asin(F.sqrt(
            F.pow(F.sin(F.radians(F.col("lat") - F.col("prev_lat")) / 2), 2) +
            F.cos(F.radians(F.col("prev_lat"))) *
            F.cos(F.radians(F.col("lat"))) *
            F.pow(F.sin(F.radians(F.col("lng") - F.col("prev_lng")) / 2), 2)
        ))
    )
    .withColumn("time_hours",
        F.greatest(
            (F.col("event_ts").cast("long") - F.col("prev_ts").cast("long")) / 3600.0,
            F.lit(1 / 3600.0)
        )
    )
    .withColumn("speed_kmh", F.round(F.col("dist_km") / F.col("time_hours"), 2))
)

impossible = df_velocity.filter(F.col("speed_kmh") > MAX_SPEED_KMH)
print(f"Total impossible velocity pairs: {impossible.count():,}")

Total impossible velocity pairs: 1,019


In [0]:
# ── Blocker 2a: GPS Accuracy Check ───────────────────────────────────────────────
# accuracy_m is the GPS radius of uncertainty. High accuracy_m = imprecise location.
# A "jump" between two imprecise fixes is noise, not real movement.

print("--- accuracy_m distribution across impossible pairs ---")
acc_stats = impossible.select(
    F.min("accuracy_m").alias("min"),
    F.max("accuracy_m").alias("max"),
    F.round(F.mean("accuracy_m"), 1).alias("mean"),
    F.percentile_approx("accuracy_m", 0.5).alias("median"),
    F.count(F.when(F.col("accuracy_m") > 1000,  1)).alias("over_1km"),
    F.count(F.when(F.col("accuracy_m") > 5000,  1)).alias("over_5km"),
    F.count(F.when(F.col("accuracy_m") > 10000, 1)).alias("over_10km"),
).collect()[0]

for k, v in acc_stats.asDict().items():
    print(f"  {k:<12} : {v}")

print()
reliable = impossible.filter(F.col("accuracy_m") < 100).count()
print(f"  Pairs with accuracy_m < 100m (reliable GPS) : {reliable:,}")
print("  These are the pairs most likely to represent real impossible movement.")

--- accuracy_m distribution across impossible pairs ---
  min          : 4.1
  max          : 46.7
  mean         : 27.1
  median       : 29.4
  over_1km     : 0
  over_5km     : 0
  over_10km    : 0

  Pairs with accuracy_m < 100m (reliable GPS) : 68
  These are the pairs most likely to represent real impossible movement.


In [0]:
# ── Blocker 2b: Concentration Check ─────────────────────────────────────────────────
# Concentrated in a few device_id / app_version  → likely a bug in that version
# Spread thin across many customers/devices       → more likely to be real signal

print("--- By customer_id (top 20) ---")
display(
    impossible.groupBy("customer_id").count()
    .orderBy(F.desc("count")).limit(20)
)

print("\n--- By device_id (top 20) ---")
display(
    impossible.groupBy("device_id").count()
    .orderBy(F.desc("count")).limit(20)
)

print("\n--- By app_version ---")
display(
    impossible.groupBy("app_version")
    .agg(F.count("*").alias("pairs"), F.countDistinct("customer_id").alias("distinct_customers"))
    .orderBy(F.desc("pairs"))
)

print("\n--- By OS ---")
display(
    impossible.groupBy("os")
    .agg(F.count("*").alias("pairs"), F.countDistinct("customer_id").alias("distinct_customers"))
    .orderBy(F.desc("pairs"))
)

--- By customer_id (top 20) ---


customer_id,count
CUST-0022482,2
CUST-0044236,2
CUST-0042169,2
CUST-0011839,2
CUST-0048335,2
CUST-0007805,2
CUST-0035519,2
CUST-0042752,2
CUST-0017762,2
CUST-0035136,2



--- By device_id (top 20) ---


device_id,count
5b61d5b2-dbe,2
c13cc529-084,2
f1881190-d0b,2
4a980119-399,2
bffe23cf-08f,2
80970080-a4a,2
b501f6ab-206,2
79537ab4-566,2
13974a8c-307,2
76e97b00-63d,2



--- By app_version ---


app_version,pairs,distinct_customers
4.3.1,169,169
4.4.0,149,149
5.0.0,149,148
4.2.0,148,148
4.2.1,146,146
4.3.0,139,139
null,119,118



--- By OS ---


os,pairs,distinct_customers
Android,522,520
iOS,497,494


In [0]:
# ── Blocker 2c: Fraud Correlation ─────────────────────────────────────────────────
# If velocity-flagged customers have a materially higher fraud rate than the rest,
# these pairs are a real signal worth preserving — not noise to be filtered.

impossible_custs = impossible.select("customer_id").distinct()

fraud_comparison = (
    df_transactions
    .select("customer_id", F.col("is_fraud_flag").cast("boolean").alias("is_fraud"))
    .join(impossible_custs.withColumn("velocity_flagged", F.lit(True)), on="customer_id", how="left")
    .withColumn("velocity_flagged", F.coalesce(F.col("velocity_flagged"), F.lit(False)))
    .groupBy("velocity_flagged")
    .agg(
        F.count("*").alias("total_txns"),
        F.count(F.when(F.col("is_fraud"), 1)).alias("fraud_txns"),
    )
    .withColumn("fraud_rate_%", F.round(F.col("fraud_txns") / F.col("total_txns") * 100, 4))
)

print("Fraud rate: velocity-flagged customers vs rest")
display(fraud_comparison)
print()
print("  Materially higher rate in velocity_flagged=True → preserve as fraud feature.")
print("  No difference                                  → likely GPS noise; filter with accuracy_m threshold.")

Fraud rate: velocity-flagged customers vs rest


velocity_flagged,total_txns,fraud_txns,fraud_rate_%
false,515298,9279,1.8007
true,9994,187,1.8711



  Materially higher rate in velocity_flagged=True → preserve as fraud feature.
  No difference                                  → likely GPS noise; filter with accuracy_m threshold.


## Warning: Amount Outliers (573 transactions, top 0.1%)

Transactions above the 99.9th percentile by amount. Lower severity than the blockers but worth investigating before sign-off.

Investigation sequence:
1. **Eyeball top 30** — round numbers (10000.00, 50000.00) suggest test/synthetic data; organic-looking decimals suggest real high-value transactions
2. **By txn_type and channel** — wire transfers and B2B payments naturally have higher ceilings than card-present retail; if concentrated there, this is expected variance
3. **By currency** — unconverted JPY/IDR/KRW values look enormous relative to USD but aren’t; check if outliers cluster in specific currencies

In [0]:
# ── Amount Outliers Setup ───────────────────────────────────────────────────────────
amount_col = _double("amount")
p999 = df_transactions.select(F.percentile_approx(amount_col, 0.999).alias("p999")).collect()[0]["p999"]
print(f"99.9th percentile threshold : {p999}")

outliers = (
    df_transactions
    .withColumn("amount_d", amount_col)
    .filter(F.col("amount_d") > p999)
    .select("transaction_id", "customer_id", "merchant_id", "amount_d",
            "currency", "txn_type", "channel", "status", "is_fraud_flag", "txn_timestamp")
)
print(f"Total outlier transactions  : {outliers.count():,}")

# ── Eyeball top 30, check for round numbers ──────────────────────────────────────
print("\n--- Top 30 outliers by amount ---")
display(outliers.orderBy(F.desc("amount_d")).limit(30))

# Round number check — amounts divisible by 1000 with no cents suggest test/synthetic data
total = outliers.count()
round_1000 = outliers.filter(F.col("amount_d") % 1000 == 0).count()
round_100  = outliers.filter((F.col("amount_d") % 100 == 0) & (F.col("amount_d") % 1000 != 0)).count()
print(f"\n  Amounts exactly divisible by 1000 : {round_1000:,} of {total:,}  (possible test data)")
print(f"  Amounts exactly divisible by 100  : {round_100:,} of {total:,}  (less suspicious)")

99.9th percentile threshold : 1689.98
Total outlier transactions  : 576

--- Top 30 outliers by amount ---


transaction_id,customer_id,merchant_id,amount_d,currency,txn_type,channel,status,is_fraud_flag,txn_timestamp
b636ee76-d039-485f-aa58-1167243712c8,CUST-0045864,MERC-007268,14366.62,USD,purchase,POS,flagged,true,2025-06-01T04:57:17.050Z
5933eb8b-6698-4709-8fd9-99284fffc9dd,CUST-0008436,MERC-005167,9086.42,USD,withdrawal,online,flagged,true,2026-04-05T02:04:25.050Z
9f626d8a-0278-4464-8ace-e1e6845be9f2,CUST-0029890,MERC-002069,8880.71,USD,purchase,POS,flagged,true,2026-01-04T03:27:05.050Z
b17841b1-05a7-4285-b897-36535c2245c5,CUST-0016813,MERC-000319,7844.38,USD,refund,online,flagged,true,2025-09-27T02:40:21.050Z
acd724e1-e475-48d2-ba30-58eba2d4a0b2,CUST-0007696,MERC-003366,7238.1,USD,purchase,POS,flagged,true,2025-10-01T03:45:49.050Z
005c2711-3df0-4236-bb14-24b226a34a8b,CUST-0046697,MERC-004222,6891.6,USD,refund,POS,flagged,true,2025-04-28T02:16:52.050Z
828300ff-312a-4587-a6e4-caf01d05cdc3,CUST-0027666,MERC-002163,6786.07,USD,purchase,POS,flagged,true,2025-07-23T04:06:55.050Z
2bd05c3a-e938-44b1-99d4-63eb8e960a85,CUST-0038836,MERC-004839,6683.68,USD,purchase,ATM,flagged,true,2025-12-24T03:48:35.050Z
964cbd4c-7350-4229-a2bb-b7ec50a95011,CUST-0003869,MERC-008587,6673.33,USD,purchase,POS,flagged,true,2026-03-09T02:02:42.050Z
1616f2eb-cc01-44f7-8cbe-c3ef9b97de35,CUST-0015277,MERC-007976,6647.24,USD,purchase,online,flagged,true,2025-08-29T01:24:35.050Z



  Amounts exactly divisible by 1000 : 0 of 576  (possible test data)
  Amounts exactly divisible by 100  : 0 of 576  (less suspicious)


In [0]:
# ── Amount Outliers b: Breakdown by txn_type, Channel, Currency ──────────────────

print("--- By txn_type ---")
display(
    outliers.groupBy("txn_type")
    .agg(
        F.count("*").alias("count"),
        F.round(F.min("amount_d"), 2).alias("min_amount"),
        F.round(F.max("amount_d"), 2).alias("max_amount"),
        F.round(F.mean("amount_d"), 2).alias("avg_amount"),
    )
    .orderBy(F.desc("count"))
)

print("\n--- By channel ---")
display(
    outliers.groupBy("channel")
    .agg(
        F.count("*").alias("count"),
        F.round(F.min("amount_d"), 2).alias("min_amount"),
        F.round(F.max("amount_d"), 2).alias("max_amount"),
    )
    .orderBy(F.desc("count"))
)

print("\n--- By currency ---")
display(
    outliers.groupBy("currency")
    .agg(
        F.count("*").alias("count"),
        F.round(F.mean("amount_d"), 2).alias("avg_amount"),
        F.round(F.max("amount_d"), 2).alias("max_amount"),
    )
    .orderBy(F.desc("count"))
)
print()
print("  If non-USD currencies dominate the outlier list, these may be unconverted")
print("  foreign amounts — not true outliers. Confirm with the source system owner.")

--- By txn_type ---


txn_type,count,min_amount,max_amount,avg_amount
purchase,501,1692.5,14366.62,2695.06
refund,43,1702.74,7844.38,2723.65
withdrawal,32,1703.33,9086.42,2941.82



--- By channel ---


channel,count,min_amount,max_amount
POS,315,1692.5,14366.62
online,214,1699.51,9086.42
ATM,47,1699.49,6683.68



--- By currency ---


currency,count,avg_amount,max_amount
USD,576,2710.91,14366.62



  If non-USD currencies dominate the outlier list, these may be unconverted
  foreign amounts — not true outliers. Confirm with the source system owner.
